# Chapter 13 — Microstructure and Execution

Ch12 closed the cost-aware cycle: even at the optimum threshold (`k_σ` = 2.20), the
QQQ MR strategy printed an **after-cost Sharpe of −2.97**. The cost wedge ate every
trace of the +0.88 walk-forward edge. The rescue hypothesis: instead of *paying* the
spread on entry, **post a passive limit at the favorable side of the touch and earn
it instead**. If the rescue works, the strategy survives. If it doesn't — that's the
canonical microstructure result: **adverse selection eats the spread credit**.

Spec: `docs/superpowers/specs/2026-05-12-ch13-microstructure-execution-design.md`

## §1 — The rescue hypothesis

From Ch12 §2: Roll's estimator gave a half-spread `s ≈ 0.724 bp` on QQQ 1-min closes.
The lag-1 negative autocovariance Roll's exploits is *exactly* the bid-ask bounce that
Ch7's MR signal partially picks up. The unsettling implication: the Ch7 edge is partly
a measurement of *the spread the strategy must cross to capture*. Crossing it on entry
*and* exit turns a 1 bp expected reversion into a −2.2 bp net trade.

**The rescue:** post a passive buy limit at `signal_close − s` (and a passive sell
limit at `signal_close + s`). When the limit fills, the entry price is on the right
side of the touch — the spread becomes a **credit**, not a cost. Round-trip swing:
+1.448 bp instead of −1.448 bp.

**The catch (preview):** passive fills are *adversely selected*. Limits fill exactly
when the market is moving against the posted side — sellers attack your bid because
they know something. The chapter's job is to measure the catch.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CACHE = Path('../06-bridge-to-intraday/data')
qqq = pd.read_parquet(CACHE / 'qqq_1min.parquet')
# Localize to NY time; carve out RTH minutes (9:30-15:59 inclusive of 9:30, exclusive of 16:00)
qqq.index = pd.to_datetime(qqq.index, utc=True).tz_convert('America/New_York')
qqq['minute_of_day'] = qqq.index.hour * 60 + qqq.index.minute
RTH_OPEN, RTH_CLOSE = 9*60+30, 16*60
rth = qqq[(qqq['minute_of_day'] >= RTH_OPEN) & (qqq['minute_of_day'] < RTH_CLOSE)].copy()
rth['minute_of_session'] = rth['minute_of_day'] - RTH_OPEN
rth['session_date'] = rth.index.normalize()
# Per-bar log return; drop the first bar of each session (overnight gap)
rth['log_ret'] = np.log(rth['close'] / rth['close'].shift(1))
sf = rth.groupby('session_date').head(1).index
rth.loc[sf, 'log_ret'] = np.nan
print(f'Sessions: {rth["session_date"].nunique()}  Bars: {len(rth):,}')

# Ch12 handoff: Roll's half-spread = 0.724 bp = 7.24e-5 in log-return units
ROLL_S = 7.24e-5
K_SIGMA = 2.20  # Ch12 cost-aware k*
N, K = 1, 3
print(f'ROLL_S = {ROLL_S*1e4:.3f} bp   K_SIGMA = {K_SIGMA}   N={N}  K={K}')

Sessions: 251  Bars: 95,318
ROLL_S = 0.724 bp   K_SIGMA = 2.2   N=1  K=3


## §2 — Order types: a short tour

| Order type | What it does | Who pays/earns the spread | Fill certainty |
|---|---|---|---|
| **Market** | Buy/sell now at the best available price. | You cross — you're the *taker*. | Near-guaranteed |
| **Limit** | Buy at `≤ p` / sell at `≥ p`. | You join the book — you're the *maker* (if posted away from the touch). | Conditional |
| **Marketable limit** | Limit with price already at-or-through the touch. | Taker; price-protected market. | Near-guaranteed |
| **IOC (immediate-or-cancel)** | Fill what you can right now; cancel the rest. | Maker on filled portion; nothing left resting. | Partial |
| **FOK (fill-or-kill)** | All-or-nothing instant fill. | Maker if filled. | Binary |
| **Hidden** | Trades at displayed prices but invisible until executed. | Variable. | Conditional |
| **Pegged** | Limit price tracks the bid/offer at some offset. | Maker. | Conditional |

**Maker / taker.** Exchanges charge takers and pay makers a small per-share rebate.
NASDAQ's tier-1 maker rebate is ~$0.0030/share. At QQQ ≈ $695 that's 0.043 bp/leg —
small, but non-zero, and only available to maker (passive) executions.

**Payment for order flow (PFOF).** A retail "market order" through Schwab/Fidelity
is routed to a wholesaler (Citadel, Virtu, …) who fills it at-or-better than NBBO in
exchange for paying the broker. The price improvement is real (a fraction of the
spread) but the routing isn't yours, and the wholesaler earns whatever it doesn't pass
through. "Zero commission" is a billing convention, not a free lunch.

## §3 — The limit-order simulation

**Fill rule.** A passive entry posted at the signal bar's close fills if and only if
a *subsequent* bar within the K-bar hold window prints at-or-through the limit:

- Buy limit at `signal_close − s` fills if any later bar's `low ≤ signal_close − s`.
- Sell limit at `signal_close + s` fills if any later bar's `high ≥ signal_close + s`.

If neither happens within K bars, the limit *expires unfilled*. We record those signals
anyway — they are critical for the §4 toxicity diagnostic.

**Exit rule.** Unchanged from Ch12: market-on-open at bar `entry + K`. The exit still
crosses the spread — the chapter measures whether earning the spread on entry is enough.

In [2]:
# Pre-compute per-session signal-window prior returns for the rolling σ estimate.
# The signal window is minutes 360-389 of session (Ch7 closing window, last 30 min).
def precompute_priors(rth_df):
    priors = {}
    dates = sorted(rth_df['session_date'].unique())
    for date in dates:
        day = rth_df[rth_df['session_date'] == date].sort_index().reset_index()
        pts = []
        for i in range(N, len(day)):
            if 360 <= day.loc[i, 'minute_of_session'] < 390 - K:
                pts.append(np.log(day.loc[i, 'close'] / day.loc[i-N, 'close']))
        priors[date] = np.array(pts)
    return priors, dates

priors_by_session, dates_sorted = precompute_priors(rth)

def backtest(rth_df, entry_mode='market', limit_offset=ROLL_S, lookback=20):
    """Event-driven backtest with market or limit entries; K-bar hold.
    Returns a trade ledger with: filled flag, pnl_log, and (for unfilled-limit
    rows) drift_K_unconditional — the post-signal drift in strategy direction.
    """
    trades = []
    for date_idx, date in enumerate(dates_sorted):
        if date_idx < lookback:  # need σ history
            continue
        # Trailing 20-session pool of prior N-bar returns → today's σ threshold
        history = np.concatenate([priors_by_session[d]
                                  for d in dates_sorted[date_idx-lookback:date_idx]])
        thresh = K_SIGMA * history.std() if len(history) > 0 else None
        if thresh is None:
            continue
        day = rth_df[rth_df['session_date'] == date].sort_index().reset_index()
        in_pos, entry_i, entry_px = 0, None, None
        pending_limit, pending_sig, pending_signal_i = None, 0, None
        for i in range(len(day)):
            mos = day.loc[i, 'minute_of_session']
            # Pending limit: check this bar's H/L for a touch
            if pending_limit is not None and i > pending_signal_i:
                bar_high = day.loc[i, 'high']
                bar_low = day.loc[i, 'low']
                if pending_sig == +1 and bar_low <= pending_limit:
                    in_pos, entry_i, entry_px = +1, i, pending_limit
                    pending_limit = None
                elif pending_sig == -1 and bar_high >= pending_limit:
                    in_pos, entry_i, entry_px = -1, i, pending_limit
                    pending_limit = None
                elif i - pending_signal_i >= K:
                    # Limit expired — record an unfilled row with the K-bar drift
                    # in strategy direction for the toxicity diagnostic.
                    sig_i = pending_signal_i
                    end_i = min(sig_i + K, len(day) - 1)
                    drift_uncond = np.log(day.loc[end_i, 'close'] /
                                          day.loc[sig_i, 'close']) * pending_sig
                    trades.append({'date': date, 'filled': False, 'pnl_log': 0.0,
                                   'drift_K_unconditional': drift_uncond})
                    pending_limit, pending_sig, pending_signal_i = None, 0, None
            # Time-stop exit at K bars after entry (market on next bar's open)
            if in_pos != 0 and (i - entry_i) >= K:
                exit_i = min(i + 1, len(day) - 1)
                exit_px = day.loc[exit_i, 'open']
                pnl = in_pos * np.log(exit_px / entry_px)
                trades.append({'date': date, 'filled': True, 'pnl_log': pnl})
                in_pos, entry_i, entry_px = 0, None, None
            # Signal detection (Ch7 window, in_pos and pending checks)
            if in_pos == 0 and pending_limit is None and \
               360 <= mos < 390 - K and i >= N:
                prior_N = np.log(day.loc[i, 'close'] / day.loc[i-N, 'close'])
                if abs(prior_N) > thresh:
                    # MR: buy after downward overshoot, sell after upward
                    sig = -1 if prior_N > 0 else +1
                    if entry_mode == 'market':
                        enxt = min(i + 1, len(day) - 1)
                        in_pos, entry_i, entry_px = sig, i, day.loc[enxt, 'open']
                    else:
                        # Buy below signal close; sell above
                        pending_limit = day.loc[i, 'close'] - sig * limit_offset
                        pending_sig, pending_signal_i = sig, i
    return pd.DataFrame(trades)

In [3]:
# Market-order baseline (Ch12 cost-aware k*, pre-cost)
tr_market = backtest(rth, entry_mode='market')
# Passive limit at-the-touch
tr_limit = backtest(rth, entry_mode='limit', limit_offset=ROLL_S)
filled = tr_limit[tr_limit['filled']].copy()
unfilled = tr_limit[~tr_limit['filled']].copy()
fill_rate = len(filled) / max(len(tr_limit), 1)

def annualize_sharpe(pnl_series, n_trades, n_sessions):
    if n_trades < 2 or pnl_series.std() == 0:
        return float('nan')
    tpy = n_trades / max(n_sessions, 1) * 252  # trades/year scaling
    return (pnl_series.mean() / pnl_series.std()) * np.sqrt(tpy)

sessions_total = tr_market['date'].nunique()
sessions_filled = filled['date'].nunique()
s_market = annualize_sharpe(tr_market['pnl_log'], len(tr_market), sessions_total)
s_limit_filled = annualize_sharpe(filled['pnl_log'], len(filled), sessions_filled)

print(f'Market signals:  {len(tr_market):3d}  mean={tr_market["pnl_log"].mean()*1e4:+.3f} bp  Sharpe={s_market:+.3f}')
print(f'Limit signals:   {len(tr_limit):3d}  filled={len(filled)}  unfilled={len(unfilled)}  fill_rate={fill_rate:.3f}')
print(f'  Filled-only:   mean={filled["pnl_log"].mean()*1e4:+.3f} bp  Sharpe={s_limit_filled:+.3f}  (pre-toxicity)')

Market signals:  200  mean=+0.461 bp  Sharpe=+1.183
Limit signals:   176  filled=157  unfilled=19  fill_rate=0.892
  Filled-only:   mean=+0.288 bp  Sharpe=+0.583  (pre-toxicity)


**First impression — the rescue appears to work.** Pre-toxicity, the filled-only
Sharpe (+0.58) is a +3.5-point swing from Ch12's market-order, after-cost Sharpe
(−2.97). Fill rate is 89% — at-the-touch on 1-min bars is aggressive enough that most
signals fill within the K=3 bar window. **89% fill, +0.58 Sharpe — looks like a clean
rescue.** §4 explains why this number is a lie.

## §4 — The adverse-selection diagnostic

**Adverse selection** (a.k.a. *toxicity*) is the tendency for passive fills to occur
exactly when the market is about to move against the posted side. A passive buy limit
fills when sellers attack the bid — sometimes because the sellers know something the
passive order doesn't. The fill happens *because* bad news is being priced in.

**Diagnostic.** Compare two K-bar drifts in strategy direction:

- `drift_unconditional` — mean K-bar drift over *all* signal bars (= the market-order
  Sharpe-batch's mean PnL, since market orders fill on every signal).
- `drift_filled` — mean K-bar drift on the *subset of signals that actually filled* as
  passive limits.

$$\textsf{toxicity} = \textsf{drift}_\textsf{unconditional} - \textsf{drift}_\textsf{filled}$$

Positive toxicity means **filled trades had worse post-fill drift than the unconditional
signal universe** — i.e., adverse selection is real and consuming the spread credit.

In [4]:
drift_unc_bp = tr_market['pnl_log'].mean() * 1e4
drift_filled_bp = filled['pnl_log'].mean() * 1e4
toxicity_bp = drift_unc_bp - drift_filled_bp
drift_unfilled_bp = unfilled['drift_K_unconditional'].mean() * 1e4

print(f'drift_unconditional (all market-order signals): {drift_unc_bp:+.4f} bp')
print(f'drift_filled        (limit fills only):         {drift_filled_bp:+.4f} bp')
print(f'toxicity = unc − filled:                        {toxicity_bp:+.4f} bp')
print(f'drift on UNFILLED signals (strategy dir):       {drift_unfilled_bp:+.4f} bp')

# Cost stack from Ch12: half-spread = 0.724 bp/leg; commission = 0.072 bp/leg;
# impact (consolidated ADV at $30k) = 0.014 bp/leg. Entry leg pays NO spread (it IS
# the passive fill). Exit leg crosses at next bar's open → pays one half-spread + comm + impact.
HALF_SPREAD_BP = 0.724
COMM_RT_BP = 0.144  # 2 × 0.072
IMPACT_RT_BP = 0.029  # 2 × 0.014 at consolidated ADV (Ch12)
exit_leg_cost_bp = HALF_SPREAD_BP + IMPACT_RT_BP/2 + COMM_RT_BP/2
# (Entry-leg commission still applies; we charge it on the entry side too.)
filled['pnl_net_log'] = filled['pnl_log'] - (exit_leg_cost_bp + COMM_RT_BP/2) / 1e4
s_limit_post = annualize_sharpe(filled['pnl_net_log'], len(filled), sessions_filled)
print(f'\nSharpe (limit, post-toxicity, net of exit spread + commission): {s_limit_post:+.3f}')

drift_unconditional (all market-order signals): +0.4612 bp
drift_filled        (limit fills only):         +0.2882 bp
toxicity = unc − filled:                        +0.1730 bp
drift on UNFILLED signals (strategy dir):       +8.1060 bp

Sharpe (limit, post-toxicity, net of exit spread + commission): -1.202


**Two findings sitting next to each other.**

1. **Toxicity is small (+0.17 bp).** The filled subset has only slightly weaker drift
   than the unconditional universe. By the strict toxicity definition the rescue *partly*
   survives — the spread credit isn't fully cancelled by adverse selection on this strategy.
2. **But the unfilled signals had +8.11 bp of drift in strategy direction.** The signals
   the passive limit *missed* are the ones where the price ran the strategy's way. The
   asymmetry is the textbook microstructure story in its starkest form: **the biggest
   winners self-cancel because the passive offer is never lifted**.

Once we re-apply the exit-leg costs (the exit still crosses, paying half-spread +
commission + impact), the filled-only Sharpe drops from +0.58 to **−1.20**. The rescue
narrowed the gap from −2.97 toward zero, but the exit side of the trade still pays the
wedge that killed Ch12.

## §5 — Latency

**Signal-to-fill latency** = wall-clock time between the signal logic emitting an
order and the exchange matching it. Components: network RTT, broker processing,
exchange matching, cancel/replace cycles.

| Regime | Typical latency |
|---|---|
| Co-located HFT | < 1 ms |
| Cloud-hosted retail bot | 5–50 ms |
| Retail desktop on home internet | 50–500 ms |

**Cost framing.** During the latency window the price drifts. If price moves
independently of the signal (random-walk regime), the standard deviation of the drift is

$$\sigma_{\textsf{drift}} = \sigma_{1\textsf{min}} \cdot \sqrt{\frac{\Delta t}{60 s}}$$

where:
- σ_{1min} = per-minute standard deviation of log-returns (QQQ ≈ 4.3 bp)
- Δt = latency in seconds
- 60 s = the reference bar duration used to estimate σ_{1min}

Critically, σ_drift **does not depend on the bar resolution** — only on the latency
duration. What changes with bar resolution is the *ratio* of σ_drift to the typical
bar's own σ. At 1-min/100ms the ratio is 4%; at 1-sec/200ms it's 45%.

In [5]:
sigma_per_min_bp = rth['log_ret'].std() * 1e4  # QQQ per-bar (1-min) σ in bp
print(f'QQQ 1-min σ: {sigma_per_min_bp:.3f} bp')

rows = []
for bar_s, lat_ms in [(60, 100), (60, 500), (15, 200), (5, 200), (1, 200)]:
    # σ over the latency window — note: depends only on lat_ms, not bar_s
    sigma_drift_bp = sigma_per_min_bp * np.sqrt((lat_ms/1000) / 60)
    e_abs_drift_bp = sigma_drift_bp * np.sqrt(2 / np.pi)  # E|N(0,σ)| = σ√(2/π)
    sigma_bar_bp = sigma_per_min_bp * np.sqrt(bar_s / 60)  # σ at this bar resolution
    rows.append({'bar': f'{bar_s}s', 'latency_ms': lat_ms,
                 'σ_drift_bp': round(sigma_drift_bp, 4),
                 'E|drift|_bp': round(e_abs_drift_bp, 4),
                 'drift_σ / bar_σ': round(sigma_drift_bp / sigma_bar_bp, 3)})
lat_table = pd.DataFrame(rows)
print(lat_table.to_string(index=False))

QQQ 1-min σ: 4.312 bp
bar  latency_ms  σ_drift_bp  E|drift|_bp  drift_σ / bar_σ
60s         100      0.1760       0.1405            0.041
60s         500      0.3936       0.3141            0.091
15s         200      0.2489       0.1986            0.115
 5s         200      0.2489       0.1986            0.200
 1s         200      0.2489       0.1986            0.447


**Read the ratio column.** At 1-min/100ms, latency drift is 4% of a typical bar's σ —
invisible. At 1-sec/200ms it's 45% — dominant; you're paying half a bar of drift on
every trade. The strategy's bar-resolution choice (made implicitly in Ch6/Ch8 at 1-min)
is partly a latency-budget choice. A 1-sec or sub-second strategy is unviable from a
home internet connection.

## §6 — Maker rebates and the final verdict

Filled passive trades earn the maker rebate on the entry leg. NASDAQ tier-1:
**$0.0030 per share**. For QQQ at ≈ $695, that's **0.043 bp per leg** — small but
non-zero.

In [6]:
QQQ_PX = float(rth['close'].iloc[-1])
rebate_bp_per_leg = 0.0030 / QQQ_PX * 1e4
print(f'QQQ ref px: ${QQQ_PX:.2f}   rebate: {rebate_bp_per_leg:.4f} bp/leg')

filled['pnl_rebate_log'] = filled['pnl_net_log'] + rebate_bp_per_leg / 1e4
s_limit_rebate = annualize_sharpe(filled['pnl_rebate_log'], len(filled), sessions_filled)
print(f'Sharpe (limit, post-toxicity + maker rebate): {s_limit_rebate:+.3f}')

QQQ ref px: $694.93   rebate: 0.0432 bp/leg
Sharpe (limit, post-toxicity + maker rebate): -1.114


### The deflation arc, closed

| Stage | Sharpe | Notes |
|---|---:|---|
| Ch11 walk-forward OOS, pre-cost | **+0.88** | Ch11 §7 (CI straddles zero) |
| Ch12 cost-aware k\*, market orders, after-cost | **−2.97** | The Ch12 verdict |
| Ch13 passive entry, filled-only, pre-toxicity (pre-cost) | **+0.58** | Apparent rescue |
| Ch13 passive entry, filled-only, post-toxicity, net of exit spread + commission | **−1.20** | Actual realized |
| Ch13 passive entry, filled-only, post-toxicity + maker rebate | **−1.11** | Final |

**The arc:** +0.88 → −2.97 → −1.11. Passive execution narrows the loss by +1.9 Sharpe
points relative to market orders, but the after-execution Sharpe is still solidly
negative. **The rescue partially works and fully fails to save the strategy.** Toxicity
in *this* strategy is small (0.17 bp); the gap that closes the rescue is the exit leg,
which still crosses the spread.

A different strategy with a larger pre-cost edge (per-trade mean ≥ 3 bp) and matched
fill rate would survive this chain. This one doesn't.

## So what?

1. **Passive execution is a tool, not a cure.** It helps when (a) the strategy is
   cost-dominated and (b) the signal has low adverse-selection sensitivity. The Ch7 MR
   strategy is cost-dominated (a) — but the exit leg still crosses, and crossing twice
   is what kills it. Pure maker/maker would need passive *exits* too, which a time-stop
   strategy can't do without abandoning the time stop.
2. **Diagnose toxicity before celebrating the spread credit.** Filled-only conditional
   drift is the diagnostic. The pre-toxicity Sharpe is the version of the strategy that
   doesn't exist.
3. **Match latency to bar resolution.** A 1-sec strategy at 200ms latency is leaking
   45% of a bar's σ on every trade. A 1-min strategy at 200ms latency is fine. The
   resolution choice in Ch6/Ch8 is partly a latency-budget choice.
4. **The unfilled bucket carries information.** When passive limits *don't* fill, the
   price ran your way — average drift in strategy direction was +8.1 bp on the 19
   unfilled signals here. The passive offer self-cancels the strategy's biggest wins.

## Key Terms

Market order · Limit order · IOC · FOK · Maker · Taker · Payment for order flow ·
Fill rate · Adverse selection (toxicity) · Latency · Maker rebate. See `glossary.md`.

## Up next

**Ch14 — Position sizing & risk of ruin.** With the after-execution expectancy in hand
(here, negative), Kelly tells us how much capital to deploy. Spoiler: when expectancy
is negative, Kelly says *zero*. But the framework matters for the next strategy.

## Exercises

1. **Fill-rule strictness.** Re-run the §3 backtest under three fill rules:
   (a) any later bar within K touches the limit (used in the chapter),
   (b) only the *next bar's open* may touch,
   (c) a bar must *trade through* the limit by ε ticks ($0.01 say).
   Compute fill rate, drift_filled, and toxicity under each. Does the +0.17 bp toxicity
   result survive rule (c)? Which rule is the most realistic for a retail simulator,
   and why?
2. **Latency sensitivity sweep.** For QQQ at 1-min, 15-sec (resampled), and 5-sec
   resolutions, compute σ<sub>drift</sub> and E\|drift\| under 100 ms and 500 ms latency assumptions.
   At what bar resolution does the latency drift exceed the 0.72 bp half-spread credit?
   *Hint:* you don't need new data; σ<sub>1min</sub> and the √(Δt/60) formula are sufficient.
3. **Maker-rebate breakeven.** Hold fill rate (0.89) and toxicity (0.17 bp) constant.
   What per-trade gross PnL (in bp) would the strategy need before the 0.043 bp/leg
   maker rebate is the *deciding* factor between losing and breaking even on after-cost
   Sharpe? Express the answer in terms of fill rate, gross per-trade std, and the
   exit-leg cost stack from Ch12.